In [1]:
import torch
from data_handler import DataHandlerModule
import json
from model_handler import ModelHandlerModule

config = {
    "seed":           2,
    "data_name":      "ieee_cis",
    "raw_dir":        "./data",  # where CSVs live
    "sample":         10000,                                  # small sample to verify
    "multi_relation": True,
    "n_head":         [2, 2],
    "n_head_agg":     8,
    "feat_drop":      0,
    "attn_drop":      0,
    "train_ratio":    0.1,
    "test_ratio":     0.67,
    "emb_size":       [64, 64],
    "lr":             0.01,
    "weight_decay":   0.001,
    "epochs":         50,           # small for quick check
    "valid_epochs":   10,
    "batch_size":     1024,
    "patience":       20,
    "cuda_id":        0,
    "save_dir":       "./results/ieee_cis"
}

In [2]:
data_handler  = DataHandlerModule(config)
model_handler = ModelHandlerModule(config, data_handler)
model_handler.train()

drag_model = model_handler.model
graph      = data_handler.dataset['graph']

Loading and preprocessing the dataset ieee_cis...
[IEEE-CIS] Loading from ./data ...
  Merged:    (590540, 434)  |  Fraud rate: 0.0350
  Sampled:   500000 rows  (fraud=17495, legit=482505)
  print:     500,000
  Etypes:    ['addr_link', 'card_link', 'time_link']
  Features:  50
  Labels:    [482505  17495]
Finished data loading and preprocessing!

 ********************  Train the DRAG  ********************
Epoch: 1 (Best: 0), loss: 0.0008773392154215008, time: 0.24160289764404297s
Epoch: 2 (Best: 0), loss: 0.0008374864936460279, time: 0.08865642547607422s
Epoch: 3 (Best: 0), loss: 0.0008028203494483431, time: 0.08894777297973633s
Epoch: 4 (Best: 0), loss: 0.0007703910352200897, time: 0.08411169052124023s
Epoch: 5 (Best: 0), loss: 0.0007770790648592254, time: 0.09439730644226074s
Epoch: 6 (Best: 0), loss: 0.0007453746245042464, time: 0.09127068519592285s
Epoch: 7 (Best: 0), loss: 0.0007135566405867113, time: 0.12985682487487793s
Epoch: 8 (Best: 0), loss: 0.0007349427066749902, time: 0.0

/mnt/c/Users/Goofy/Documents/Projects/IFT-6759-AP-DRAG_Augmentation/model_handler.py:161: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(self

Test performance: - Epoch_Best: 19	- F1: 0.2175	- Recall: 0.6226	- Precision: 0.1318	- Accuracy: 0.8433	- AUC-ROC: 0.8079	- F1-macro: 0.5652	- Recall-macro: 0.7369	- AP: 0.5580	



In [3]:
from post_training.contrastive_drag import run_contrastive_pipeline
import copy

# Keep the original untouched
drag_model_original = drag_model

# Each call gets its own independent copy of the baseline weights
drag_model_direct_inference = run_contrastive_pipeline(
    copy.deepcopy(drag_model_original), graph, config,
    data_handler=data_handler, run_3a=True, run_3b=False
)


Device: cuda

PHASE 2 — Contrastive Fine-tuning
[Contrastive] Batch Size: 4096
[Contrastive] Encoder embedding dim: 64
  Epoch   1/50  Loss: 7.6960
  Epoch   2/50  Loss: 7.4248
  Epoch   3/50  Loss: 7.3817
  Epoch   4/50  Loss: 7.3579
  Epoch   5/50  Loss: 7.3362
  Epoch   6/50  Loss: 7.3209
  Epoch   7/50  Loss: 7.3060
  Epoch   8/50  Loss: 7.2941
  Epoch   9/50  Loss: 7.2862
  Epoch  10/50  Loss: 7.2785
  Epoch  11/50  Loss: 7.2726
  Epoch  12/50  Loss: 7.2686
  Epoch  13/50  Loss: 7.2639
  Epoch  14/50  Loss: 7.2591
  Epoch  15/50  Loss: 7.2560
  Epoch  16/50  Loss: 7.2538
  Epoch  17/50  Loss: 7.2513
  Epoch  18/50  Loss: 7.2476
  Epoch  19/50  Loss: 7.2444
  Epoch  20/50  Loss: 7.2428
  Epoch  21/50  Loss: 7.2405
  Epoch  22/50  Loss: 7.2387
  Epoch  23/50  Loss: 7.2363
  Epoch  24/50  Loss: 7.2349
  Epoch  25/50  Loss: 7.2334
  Epoch  26/50  Loss: 7.2317
  Epoch  27/50  Loss: 7.2302
  Epoch  28/50  Loss: 7.2285
  Epoch  29/50  Loss: 7.2263
  Epoch  30/50  Loss: 7.2257
  Epoch  31

In [4]:
drag_model_finetuned = run_contrastive_pipeline(
    copy.deepcopy(drag_model_original), graph, config,
    data_handler=data_handler, run_3a=False, run_3b=True
)

Device: cuda

PHASE 2 — Contrastive Fine-tuning
[Contrastive] Batch Size: 4096
[Contrastive] Encoder embedding dim: 64
  Epoch   1/50  Loss: 7.7026
  Epoch   2/50  Loss: 7.4268
  Epoch   3/50  Loss: 7.3825
  Epoch   4/50  Loss: 7.3551
  Epoch   5/50  Loss: 7.3308
  Epoch   6/50  Loss: 7.3164
  Epoch   7/50  Loss: 7.3010
  Epoch   8/50  Loss: 7.2902
  Epoch   9/50  Loss: 7.2820
  Epoch  10/50  Loss: 7.2758
  Epoch  11/50  Loss: 7.2707
  Epoch  12/50  Loss: 7.2662
  Epoch  13/50  Loss: 7.2619
  Epoch  14/50  Loss: 7.2589
  Epoch  15/50  Loss: 7.2549
  Epoch  16/50  Loss: 7.2519
  Epoch  17/50  Loss: 7.2489
  Epoch  18/50  Loss: 7.2461
  Epoch  19/50  Loss: 7.2430
  Epoch  20/50  Loss: 7.2415
  Epoch  21/50  Loss: 7.2393
  Epoch  22/50  Loss: 7.2368
  Epoch  23/50  Loss: 7.2343
  Epoch  24/50  Loss: 7.2330
  Epoch  25/50  Loss: 7.2308
  Epoch  26/50  Loss: 7.2289
  Epoch  27/50  Loss: 7.2276
  Epoch  28/50  Loss: 7.2261
  Epoch  29/50  Loss: 7.2244
  Epoch  30/50  Loss: 7.2244
  Epoch  31